In [11]:
# Cell 1: Install (if needed) and imports
# Run this cell first. On most systems you already have these libs.
!pip install --quiet opencv-python-headless numpy

import cv2
import numpy as np
from pathlib import Path
from math import cos, sin
from typing import Tuple, Optional


In [12]:
# Cell 2: small utilities: draw_point, draw_line, intersect
def draw_point(img, point, color=(0,0,255)):
    if point is None:
        return img
    return cv2.circle(img, (int(point[0]), int(point[1])), radius=5, color=color, thickness=5)

def draw_line(img, line, color=(0,0,255)):
    if line is None:
        return img
    rho, theta = line
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    pt1 = (int(x0 + 2000 * (-b)), int(y0 + 2000 * (a)))
    pt2 = (int(x0 - 2000 * (-b)), int(y0 - 2000 * (a)))
    return cv2.line(img, pt1, pt2, color, 3, cv2.LINE_AA)

def intersect(line1, line2):
    if line1 is None or line2 is None:
        return None
    rho1, th1 = line1
    rho2, th2 = line2
    if abs(th1 - th2) < 1e-6:
        return None
    u = (rho1 * np.sin(th2) - rho2 * np.sin(th1)) / np.sin(th2 - th1)
    if abs(np.sin(th1)) > 1e-6:
        v = (rho1 - u * np.cos(th1)) / np.sin(th1)
    else:
        v = (rho2 - u * np.cos(th2)) / np.sin(th2)
    return [int(u), int(v)]


In [13]:
# Cell 3: KeyPoints class + world coordinates used for PnP
# World coordinate conventions (x right, y forward, z up). Distances in meters.

# central circle offsets (meters)
right_circle_world = [9.15, 0, 0]
left_circle_world  = [-9.15, 0, 0]
behind_circle_world = [0, 0, 9.15]
front_circle_world  = [0, 0, -9.15]
front_middle_line_world = [0, 0, -34]
back_middle_line_world  = [0, 0, 34]

corner_back_left_world   = [-52.5, 0, 34]
corner_front_left_world  = [-52.5, 0, -34]
corner_back_right_world  = [52.5, 0, 34]
corner_front_right_world = [52.5, 0, -34]

DIST_TO_CENTER = 77.0

class KeyPoints:
    def __init__(self):
        self.right_circle = None
        self.left_circle = None
        self.behind_circle = None
        self.front_circle = None
        self.front_middle_line = None
        self.back_middle_line = None
        self.corner_back_left = None
        self.corner_back_right = None
        self.corner_front_left = None
        self.corner_front_right = None

    def draw(self, img):
        for p in (self.right_circle, self.left_circle, self.behind_circle, self.front_circle,
                  self.front_middle_line, self.back_middle_line,
                  self.corner_back_left, self.corner_back_right, self.corner_front_left, self.corner_front_right):
            img = draw_point(img, p)
        return img

    def make_2d_3d_association_list(self):
        pixels, points_world = [], []
        if self.right_circle is not None:
            pixels.append(self.right_circle); points_world.append(right_circle_world)
        if self.left_circle is not None:
            pixels.append(self.left_circle); points_world.append(left_circle_world)
        if self.behind_circle is not None:
            pixels.append(self.behind_circle); points_world.append(behind_circle_world)
        if self.front_circle is not None:
            pixels.append(self.front_circle); points_world.append(front_circle_world)
        if self.front_middle_line is not None:
            pixels.append(self.front_middle_line); points_world.append(front_middle_line_world)
        if self.back_middle_line is not None:
            pixels.append(self.back_middle_line); points_world.append(back_middle_line_world)
        if self.corner_front_left is not None:
            pixels.append(self.corner_front_left); points_world.append(corner_front_left_world)
        if self.corner_front_right is not None:
            pixels.append(self.corner_front_right); points_world.append(corner_front_right_world)
        if self.corner_back_left is not None:
            pixels.append(self.corner_back_left); points_world.append(corner_back_left_world)
        if self.corner_back_right is not None:
            pixels.append(self.corner_back_right); points_world.append(corner_back_right_world)

        pixels = np.array(pixels, dtype=np.float32)
        points_world = np.array(points_world, dtype=np.float32)
        return pixels, points_world

    def compute_focal_length(self, guess_fx):
        # If both circle edges available, estimate fx from separation
        if self.right_circle is None and self.left_circle is None:
            return guess_fx
        if self.right_circle is not None and self.left_circle is not None:
            fx = ((self.right_circle[0] - self.left_circle[0]) * DIST_TO_CENTER /
                  (right_circle_world[0] - left_circle_world[0]))
            return fx
        if self.behind_circle is None or self.front_circle is None:
            return guess_fx
        central = [int((self.behind_circle[0] + self.front_circle[0]) / 2),
                   int((self.behind_circle[1] + self.front_circle[1]) / 2)]
        if self.right_circle is None:
            fx = ((central[0] - self.left_circle[0]) * DIST_TO_CENTER / (-left_circle_world[0]))
            return fx
        if self.left_circle is None:
            fx = ((self.right_circle[0] - central[0]) * DIST_TO_CENTER / (right_circle_world[0]))
            return fx
        return guess_fx


In [14]:
# Cell 4: A simplified key-lines/key-points detector that follows your pipeline.
# It uses HoughLines and floodfill for central circle like your code but simplified.
def find_back_front_lines(img_gray):
    dst = cv2.Canny(img_gray, 50, 200, None, 3)
    lines = cv2.HoughLines(dst, 1, np.pi/180/4, 500, None, min_theta=80/180*np.pi, max_theta=100/180*np.pi)
    if lines is None:
        return None, None
    height, width = img_gray.shape[:2]
    back_line = None; front_line = None
    back_y = 0; front_y = height
    for l in lines:
        rho, theta = l[0]
        y_mid = (rho - width/2 * np.cos(theta)) / np.sin(theta)
        if back_y < y_mid < height/2:
            back_y = y_mid; back_line = (rho, theta)
        if height/2 < y_mid < front_y:
            front_y = y_mid; front_line = (rho, theta)
    return back_line, front_line

def remove_out_of_field(img, back_line, front_line):
    if back_line is None and front_line is None:
        return img
    img_copy = img.copy()
    height, width = img_copy.shape[:2]
    if back_line is None:
        rho, th = front_line
        for j in range(width):
            y = int((rho - j * np.cos(th)) / np.sin(th))
            img_copy[y:, j] = 0
        return img_copy
    if front_line is None:
        rho, th = back_line
        for j in range(width):
            y = int((rho - j * np.cos(th)) / np.sin(th))
            img_copy[:y, j] = 0
        return img_copy
    rho_b, th_b = back_line; rho_f, th_f = front_line
    for j in range(width):
        yb = int((rho_b - j * np.cos(th_b)) / np.sin(th_b))
        yf = int((rho_f - j * np.cos(th_f)) / np.sin(th_f))
        img_copy[:yb, j] = 0
        img_copy[yf:, j] = 0
    return img_copy

def find_main_line(img_gray):
    dst = cv2.Canny(img_gray, 50, 200, None, 3)
    lines = cv2.HoughLines(dst, 1, np.pi/180/2, 200, None, min_theta=0, max_theta=40/180*np.pi)
    if lines is not None:
        return tuple(lines[0][0])
    lines_other = cv2.HoughLines(dst, 1, np.pi/180/2, 250, None, min_theta=130/180*np.pi, max_theta=np.pi)
    if lines_other is not None:
        return tuple(lines_other[0][0])
    return None

def find_central_circle(img_gray, back_middle_point, front_middle_point, main_line):
    # simplified floodfill method
    if back_middle_point is None or front_middle_point is None:
        return None, None, None, None
    dst = cv2.Canny(img_gray, 20, 100, None, 3)
    h, w = dst.shape[:2]
    im_ff = cv2.dilate(dst, kernel=np.ones((7,7), np.uint8))
    mask = np.zeros((h+2, w+2), np.uint8)
    back = np.array(back_middle_point); front = np.array(front_middle_point)
    center_approx = (0.3*front + 0.7*back).astype(int)
    for seed in (-150, -100, -50, 50, 100, 150):
        root = (int(center_approx[0]) + seed, int(center_approx[1]))
        if 0 <= root[0] < w:
            if im_ff[root[1], root[0]] != 0:
                continue
            cv2.floodFill(im_ff, mask, root, 128)
    final_mask = cv2.inRange(im_ff, 127, 129)
    final_mask = cv2.dilate(final_mask, kernel=np.ones((15,15), np.uint8))
    final_mask = cv2.erode(final_mask, kernel=np.ones((10,10), np.uint8))
    cnts = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    if not cnts:
        return None, None, None, None
    c = max(cnts, key=cv2.contourArea)
    left_circle = tuple(c[c[:,:,0].argmin()][0])
    right_circle = tuple(c[c[:,:,0].argmax()][0])
    y_top = tuple(c[c[:,:,1].argmin()][0])[1]
    y_bottom = tuple(c[c[:,:,1].argmax()][0])[1]
    behind_circle = [int((main_line[0] - y_top * np.sin(main_line[1])) / np.cos(main_line[1])), y_top]
    front_circle  = [int((main_line[0] - y_bottom * np.sin(main_line[1])) / np.cos(main_line[1])), y_bottom]
    # sanity
    if left_circle[0] == 0: left_circle = None
    if right_circle[0] == img_gray.shape[1]-1: right_circle = None
    return left_circle, right_circle, behind_circle, front_circle

def find_key_points(img_bgr):
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    key_points = KeyPoints()
    back_line, front_line = find_back_front_lines(img_gray)
    img_wo = remove_out_of_field(img_bgr, back_line, front_line)
    main_line = find_main_line(cv2.cvtColor(img_wo, cv2.COLOR_BGR2GRAY))
    key_points.back_middle_line = intersect(main_line, back_line)
    key_points.front_middle_line = intersect(main_line, front_line)
    left_c, right_c, behind_c, front_c = find_central_circle(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
                                                             key_points.back_middle_line, key_points.front_middle_line, main_line)
    key_points.left_circle, key_points.right_circle, key_points.behind_circle, key_points.front_circle = left_c, right_c, behind_c, front_c
    # try to find corners from goal lines omitted for brevity; good-enough for many views
    return key_points, (back_line, front_line, main_line)


In [15]:
# Cell 5: projection / PnP / calibration helpers (adapted)
def project_to_screen(K, to_device_from_world, point_in_world):
    # point_in_world: (3,) or list
    homog = np.ones((4,1))
    homog[0:3,0] = np.array(point_in_world).reshape((3,))
    point_in_device = to_device_from_world.dot(homog)        # 4x1
    # ensure positive depth
    if point_in_device[2,0] == 0:
        return [-1,-1]
    point_in_device_div = (point_in_device / point_in_device[2,0])[0:3,0]
    point_projected = K.dot(point_in_device_div)
    return [int(point_projected[0]), int(point_projected[1])]

def find_extrinsic_intrinsic_matrices(img, guess_fx, guess_rot, guess_trans, key_points: KeyPoints):
    h, w = img.shape[0], img.shape[1]
    pixels, points_world = key_points.make_2d_3d_association_list()
    print(f"Solving PnP with {len(pixels)} points")
    fx = key_points.compute_focal_length(guess_fx)
    K = np.array([[fx, 0, w/2],[0, fx, h/2],[0,0,1]], dtype=np.float64)
    if pixels.shape[0] <= 3:
        print("Too few points to solve PnP")
        return None, K, guess_rot, guess_trans
    success, rvec, tvec = cv2.solvePnP(points_world, pixels, K, distCoeffs=None, rvec=guess_rot, tvec=guess_trans, useExtrinsicGuess=True)
    if not success:
        print("PnP failed")
        return None, K, guess_rot, guess_trans
    rot_mat = cv2.Rodrigues(rvec)[0]  # 3x3
    to_device_from_world = np.eye(4, dtype=np.float64)
    to_device_from_world[0:3,0:3] = rot_mat
    to_device_from_world[0:3,3] = tvec.reshape((3,))
    # Check camera position in world coords: camera_pos = -R^T * t
    cam_pos_world = -np.matrix(rot_mat).T * np.matrix(tvec)
    dist_center = np.linalg.norm(cam_pos_world)
    fx = float(fx)
    # sanity check (loosely following original thresholds)
    if np.isnan(cam_pos_world).any() or fx is None or dist_center < 10.0 or dist_center > 1000.0:
        print("PnP produced suspicious results; still returning but check them")
    return to_device_from_world, K, rvec, tvec

def calibrate_from_image(img, guess_fx=2000, guess_rot=np.array([[0.25,0,0]], dtype=np.float64), guess_trans=np.array([[0],[0],[80.0]], dtype=np.float64)):
    key_points, key_lines = find_key_points(img)
    # run PnP
    to_device_from_world, K, rot, trans = find_extrinsic_intrinsic_matrices(img, guess_fx, guess_rot, guess_trans, key_points)
    return key_points, key_lines, to_device_from_world, K, rot, trans


In [16]:
# Cell 6: map pixel -> world (assume ground plane y=0)
# We assume world coordinates use y as forward (but objects on field are at same y=0 plane in Z?),
# in our KeyPoints world, points are defined as [x, y, z] where y is forward (we used y=0 for pitch),
# for simplicity we unproject rays and intersect with plane z = 0 (ground). Adjust if your convention differs.

def pixel_to_world_on_ground(pixel, K, to_device_from_world, plane_z=0):
    u, v = pixel
    R = to_device_from_world[:3, :3]
    t = to_device_from_world[:3, 3]

    # ray in device coords
    ray_device = np.linalg.inv(K) @ np.array([u, v, 1])
    ray_device = ray_device.reshape(3, 1)

    # camera position in world coordinates
    cam_world = (-R.T @ t).A1 if isinstance(R, np.matrix) else (-R.T @ t)

    # direction of ray in world coords
    dir_world = R.T @ ray_device
    dir_world = dir_world.ravel()

    # intersection with plane z = plane_z
    if abs(dir_world[2]) < 1e-6:
        return None
    s = (plane_z - cam_world[2]) / dir_world[2]
    world_point = cam_world + s * dir_world
    return world_point


def local_pixel_to_meter_scale(center_pixel, K, to_device_from_world, dx_pixels=10):
    # compute world points for center_pixel and center_pixel + dx in x image direction
    p0 = pixel_to_world_on_ground(center_pixel, K, to_device_from_world)
    p1 = pixel_to_world_on_ground((center_pixel[0]+dx_pixels, center_pixel[1]), K, to_device_from_world)
    if p0 is None or p1 is None:
        return None
    dist_m = np.linalg.norm(np.array(p1) - np.array(p0))
    meters_per_pixel = dist_m / dx_pixels
    return meters_per_pixel


In [17]:
# Cell 7: camera pose helpers
def camera_position_world_from_extrinsics(to_device_from_world):
    R = to_device_from_world[0:3,0:3]
    t = to_device_from_world[0:3,3].reshape((3,1))
    cam_pos = (-np.matrix(R).T * np.matrix(t)).A1
    return cam_pos  # numpy array (3,)

def camera_motion_between(to_device_prev, to_device_cur):
    # returns translation (meters) of camera in world coords: pos_cur - pos_prev
    pos_prev = camera_position_world_from_extrinsics(to_device_prev)
    pos_cur  = camera_position_world_from_extrinsics(to_device_cur)
    return pos_cur - pos_prev


In [21]:
# Cell 8: full pipeline to process video and output annotated file
# Usage: adjust input_path and output_path and run. This will:
#  - sample frames
#  - calibrate on first frame where PnP succeeds
#  - compute pixel->meter map at image center
#  - compute camera motion each frame
#  - produce annotated video showing yaw/fx/scale and top-down (approx) world positions for any example pixels

def process_video(input_path: str, output_path: str, calibrate_every_n_frames: int = 30, zoom_smooth: float = 0.9):
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h))

    # initial guesses
    guess_fx = 2000.0
    guess_rot = np.array([[0.25, 0.0, 0.0]], dtype=np.float64)
    guess_trans = np.array([[0.0], [0.0], [80.0]], dtype=np.float64)

    to_device_prev = None
    cam_pos_prev = None
    meters_per_pixel_center = None
    prev_scale = None
    zoom_factor = 1.0
    zoom_change = 0.0  # track relative zoom change

    frame_idx = 0
    first_good = False

    print("Processing video... this may take a while")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_disp = frame.copy()

        # --- CALIBRATION STEP ---
        if frame_idx % calibrate_every_n_frames == 0 or not first_good:
            print(f"[frame {frame_idx}] Running calibration...")
            key_points, key_lines, to_device, rot, K, trans = None, None, None, None, None, None
            try:
                key_points, key_lines, to_device, K, rot, trans = calibrate_from_image(
                    frame,
                    guess_fx=guess_fx,
                    guess_rot=guess_rot,
                    guess_trans=guess_trans
                )
            except Exception as e:
                print("Calibration exception:", e)
                to_device = None

            if to_device is not None:
                first_good = True
                to_device_prev = to_device.copy()
                cam_pos_prev = camera_position_world_from_extrinsics(to_device)

                meters_per_pixel_center = local_pixel_to_meter_scale(
                    (w / 2, h / 2), K, to_device, dx_pixels=20
                )

                # --- [ZOOM HANDLING START] ---
                if prev_scale is not None and meters_per_pixel_center is not None:
                    ratio = meters_per_pixel_center / prev_scale
                    zoom_factor = zoom_smooth * zoom_factor + (1 - zoom_smooth) * ratio
                    zoom_change = abs(1 - ratio)
                prev_scale = meters_per_pixel_center
                adjusted_scale = meters_per_pixel_center / zoom_factor
                # --- [ZOOM HANDLING END] ---

                # --- [COLOR CODE ZOOM STATUS] ---
                if zoom_change < 0.02:
                    color = (0, 255, 0)     # green = stable
                elif zoom_change < 0.08:
                    color = (0, 165, 255)   # orange = mild zoom
                else:
                    color = (0, 0, 255)     # red = strong zoom
                # --- [COLOR CODE END] ---

                # draw detected keypoints and info
                frame_disp = key_points.draw(frame_disp)
                try:
                    cv2.putText(
                        frame_disp,
                        f"Fx={K[0,0]:.1f} | scale={adjusted_scale:.4f} m/px | zoom={zoom_factor:.3f}x",
                        (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.0,
                        color,
                        2,
                    )
                except Exception:
                    pass

                guess_fx = K[0, 0]
                guess_rot = rot if rot is not None else guess_rot
                guess_trans = trans if trans is not None else guess_trans

        else:
            # --- FAST RECALIBRATION ---
            try:
                key_points, key_lines, to_device, K, rot, trans = calibrate_from_image(
                    frame,
                    guess_fx=guess_fx,
                    guess_rot=guess_rot,
                    guess_trans=guess_trans
                )
                if to_device is not None:
                    cam_pos_cur = camera_position_world_from_extrinsics(to_device)
                    cam_motion = cam_pos_cur - cam_pos_prev
                    cam_pos_prev = cam_pos_cur

                    meters_per_pixel_center = local_pixel_to_meter_scale(
                        (w / 2, h / 2), K, to_device, dx_pixels=20
                    )

                    # --- [ZOOM HANDLING DURING RE-CALIBRATION] ---
                    if prev_scale is not None and meters_per_pixel_center is not None:
                        ratio = meters_per_pixel_center / prev_scale
                        zoom_factor = zoom_smooth * zoom_factor + (1 - zoom_smooth) * ratio
                        zoom_change = abs(1 - ratio)
                    prev_scale = meters_per_pixel_center
                    adjusted_scale = meters_per_pixel_center / zoom_factor
                    # --- [ZOOM HANDLING END] ---

                    # --- [COLOR CODE ZOOM STATUS] ---
                    if zoom_change < 0.02:
                        color = (0, 255, 0)
                    elif zoom_change < 0.08:
                        color = (0, 165, 255)
                    else:
                        color = (0, 0, 255)
                    # --- [COLOR CODE END] ---

                    cv2.putText(
                        frame_disp,
                        f"Scale={adjusted_scale:.4f} m/px | zoom={zoom_factor:.3f}x",
                        (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.0,
                        color,
                        2,
                    )

                    cv2.putText(
                        frame_disp,
                        f"cam_motion: dx={cam_motion[0]:.2f}m dy={cam_motion[1]:.2f}m dz={cam_motion[2]:.2f}m",
                        (30, 100),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (0, 255, 255),
                        2,
                    )

                    frame_disp = key_points.draw(frame_disp)
                    guess_fx = K[0, 0]
                    guess_rot = rot if rot is not None else guess_rot
                    guess_trans = trans if trans is not None else guess_trans
            except Exception:
                pass

        # --- Example demo points ---
        example_pixels = [(int(w * 0.4), int(h * 0.5)), (int(w * 0.6), int(h * 0.5))]
        if "to_device" in locals() and to_device is not None:
            for pix in example_pixels:
                wp = pixel_to_world_on_ground(pix, K, to_device, plane_z=0.0)
                if wp is not None:
                    cv2.circle(frame_disp, pix, 6, (255, 0, 0), -1)
                    cv2.putText(
                        frame_disp,
                        f"{wp[0]:.1f},{wp[1]:.1f}m",
                        (pix[0] + 8, pix[1] - 8),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        1,
                    )

        out.write(frame_disp)
        frame_idx += 1

    cap.release()
    out.release()
    print("✅ Done. Output saved to", output_path)




In [24]:
# Load video and extract frames with skipping
video_path = "/Users/alanpehz/Documents/Personal/True Computer Vision/FootballTracker/Videos/demo2.mp4"
video_path_2 = "/Users/alanpehz/Documents/Personal/True Computer Vision/FootballTracker/Videos/demo1.mp4"

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps if fps > 0 else 30

print(f"Video info: {total_frames} frames, {fps:.1f} FPS, {duration:.1f}s duration")

# Skip every 5 frames to reduce processing while covering full video
frame_skip = 5
frames = []
frame_indices = []

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_skip == 0:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        frame_indices.append(frame_count)

    frame_count += 1

cap.release()

print(f"Loaded {len(frames)} frames (every {frame_skip}th frame)")
print(f"Frame shape: {frames[0].shape}")
print(f"Time coverage: 0s to {frame_indices[-1]/fps:.1f}s")


Video info: 750 frames, 25.0 FPS, 30.0s duration
Loaded 150 frames (every 5th frame)
Frame shape: (1080, 1920, 3)
Time coverage: 0s to 29.8s


In [26]:
process_video(video_path_2, "output_annotated_2.mp4", calibrate_every_n_frames=30)

Processing video... this may take a while
[frame 0] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 1] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 2] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 3] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 4] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 5] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 6] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 7] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 8] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 9] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP
[frame 10] Running calibration...
Solving PnP with 0 points
Too few points to solve PnP


In [23]:
process_video(video_path, "output_annotated.mp4", calibrate_every_n_frames=30)

Processing video... this may take a while
[frame 0] Running calibration...
Solving PnP with 1 points
Too few points to solve PnP
[frame 1] Running calibration...
Solving PnP with 1 points
Too few points to solve PnP
[frame 2] Running calibration...
Solving PnP with 1 points
Too few points to solve PnP
[frame 3] Running calibration...
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 1 points
Too few points to solve PnP
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP with 6 points
Solving PnP 

# Cell 9 (markdown style): How to integrate a tracker

1. Run `process_video(...)` to generate an annotated output and to confirm calibration quality.
2. Replace the `example_pixels` list in the pipeline with the pixel coordinates output by your ball/player tracker for each frame (e.g. [(x,y), ...]).
3. For each detected pixel, call `pixel_to_world_on_ground(pixel, K, to_device, plane_z=0.0)` to get (x,y,z) in meters on the pitch.
   - That yields world positions in a *camera-local* world frame. If you want a global stable frame across the whole video, accumulate/subtract camera motion:
       * Keep `cam_pos_prev` and add the sequence of camera motions between frames to create a stable origin.
       * For a tracked object measured at frame `t`, its global position = measured_world_pos + cumulative_camera_translation_up_to_t
4. Use differences of global positions across time (and FPS) to compute speeds (m/s).
